<a href="https://colab.research.google.com/github/valanchick/Multi-Stage-E-Commerce-Recommender-System/blob/main/E_Commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from datetime import datetime, timedelta

"""
np.random.seed(42)
n_interactions = 100000
start_date = datetime(2023, 10, 1)
timestamps = [start_date + timedelta(seconds=np.random.randint(0, 2592000)) for _ in range(n_interactions)]
timestamps.sort()

df = pd.DataFrame({
    'event_time': [t.strftime('%Y-%m-%d %H:%M:%S UTC') for t in timestamps],
    'event_type': np.random.choice(['view', 'cart', 'remove_from_cart', 'purchase'], p=[0.75, 0.15, 0.05, 0.05], size=n_interactions),
    'product_id': np.random.randint(1000000, 1015000, n_interactions),
    'category_id': np.random.randint(2053013552259662037, 2053013552259662100, n_interactions),
    'category_code': np.random.choice(['electronics.smartphone', 'appliances.kitchen', 'apparel.shoes', np.nan], p=[0.4, 0.3, 0.2, 0.1], size=n_interactions),
    'brand': np.random.choice(['apple', 'samsung', 'xiaomi', 'nike', np.nan], p = [0.3, 0.2, 0.2, 0.1, 0.2], size=n_interactions),
    'price': np.round(np.random.exponential(150, n_interactions)+10, 2),
    'user_id': np.random.randint(500000000, 500005000, n_interactions),
    'user_session': [f"session_{np.random.randint(100000)}" for _ in range(n_interactions)]
})
"""

import os
import kagglehub

dataset_dir = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")

csv_files = [f for f in os.listdir(dataset_dir) if f.endswith('.csv')]
print(f"Найдены файлы: {csv_files}")

csv_path = os.path.join(dataset_dir, csv_files[0])
print(f"Читаем файл: {csv_path}")

df = pd.read_csv(csv_path, nrows=1000000)

df = df[df['event_type']!='remove_from_cart'].copy()
df['brand'] = df['brand'].fillna('unknown_brand')
df['category_code'] = df['category_code'].fillna('unknown_category')

encoders = {}
features_to_encode = ['user_id', 'product_id', 'brand', 'category_code']

for feat in features_to_encode:
    encoders[feat] = LabelEncoder()
    df[f'{feat}_idx'] = encoders[feat].fit_transform(df[feat])

NUM_USERS = len(encoders['user_id'].classes_)
NUM_ITEMS = len(encoders['product_id'].classes_)
NUM_BRANDS = len(encoders['brand'].classes_)
NUM_CATEGORIES = len(encoders['category_code'].classes_)

print(f"Уникальных товаров: {NUM_ITEMS} | Брендов: {NUM_BRANDS} | Категорий: {NUM_CATEGORIES}")

item_features_dict = df.drop_duplicates('product_id_idx').set_index('product_id_idx')[['brand_idx', 'category_code_idx']].to_dict('index')

split_index = int(len(df)*0.8)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

test_df = test_df[test_df['user_id_idx'].isin(train_df['user_id_idx']) & test_df['product_id_idx'].isin(train_df['product_id_idx'])]

class RichEcomDataset(Dataset):
    def __init__(self, df, num_items, item_features_dict, num_negatives=4):
        self.users = df['user_id_idx'].values
        self.items = df['product_id_idx'].values
        event_weights = {'view':1.0, 'cart': 2.0, 'purchase':3.0}
        self.weights = df['event_type'].map(event_weights).values
        self.num_items = num_items
        self.num_negatives = num_negatives
        self.item_features = item_features_dict

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        i_pos = self.items[idx]
        w = self.weights[idx]

        neg_items = np.random.randint(0, self.num_items, self.num_negatives)
        all_items = np.insert(neg_items, 0, i_pos)

        brands = [self.item_features[item]['brand_idx'] for item in all_items]
        categories = [self.item_features[item]['category_code_idx'] for item in all_items]

        labels = np.zeros(self.num_negatives+1, dtype=np.float32)
        labels[0] = 1.0

        sample_weights = np.ones(self.num_negatives+1, dtype=np.float32)
        sample_weights[0] = w

        return (
            torch.tensor(u, dtype=torch.long),
            torch.tensor(all_items, dtype=torch.long),
            torch.tensor(brands, dtype=torch.long),
            torch.tensor(categories, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float32),
            torch.tensor(sample_weights, dtype=torch.float32)
        )

train_dataset = RichEcomDataset(train_df, NUM_ITEMS, item_features_dict, num_negatives=4)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

sample_u, sample_i, sample_b, sample_c, sample_y, sample_w = next(iter(train_loader))
print("Тензор Брендов (1 позитив + 4 негатива):", sample_b.shape)
print("Тензор Категорий (1 позитив + 4 негатива):", sample_c.shape)

Using Colab cache for faster access to the 'ecommerce-behavior-data-from-multi-category-store' dataset.
Найдены файлы: ['2019-Nov.csv', '2019-Oct.csv']
Читаем файл: /kaggle/input/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv
Уникальных товаров: 66001 | Брендов: 2649 | Категорий: 124
Тензор Брендов (1 позитив + 4 негатива): torch.Size([1024, 5])
Тензор Категорий (1 позитив + 4 негатива): torch.Size([1024, 5])


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class RichTwoTowerRecSys(nn.Module):
    def __init__(self, num_users, num_items, num_brands, num_cats, embed_dim=32, hidden_dim=64):
        super(RichTwoTowerRecSys, self).__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        self.brand_emb = nn.Embedding(num_brands, embed_dim)
        self.cat_emb = nn.Embedding(num_cats, embed_dim)

        self.user_net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

        self.item_net = nn.Sequential(
            nn.Linear(embed_dim*3, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, user_idx, item_idx, brand_idx, cat_idx):
        u_e = self.user_emb(user_idx)
        u_repr = self.user_net(u_e)
        u_repr = F.normalize(u_repr, p=2, dim=1)
        u_repr = u_repr.unsqueeze(1)

        i_e = self.item_emb(item_idx)
        b_e = self.brand_emb(brand_idx)
        c_e = self.cat_emb(cat_idx)

        item_concat = torch.cat([i_e, b_e, c_e], dim=2)
        i_repr = self.item_net(item_concat)
        i_repr = F.normalize(i_repr, p=2, dim=2)

        return torch.sum(u_repr * i_repr, dim=2)

model = RichTwoTowerRecSys(NUM_USERS, NUM_ITEMS, NUM_BRANDS, NUM_CATEGORIES, 32, 64)
print(model)

test_out = model(sample_u, sample_i, sample_b, sample_c)
print(f"\nРазмерность выхода модели: {test_out.shape} (Должно быть[1024, 5])")

RichTwoTowerRecSys(
  (user_emb): Embedding(170337, 32)
  (item_emb): Embedding(66001, 32)
  (brand_emb): Embedding(2649, 32)
  (cat_emb): Embedding(124, 32)
  (user_net): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
  (item_net): Sequential(
    (0): Linear(in_features=96, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
)

Размерность выхода модели: torch.Size([1024, 5]) (Должно быть[1024, 5])


In [8]:
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Обучение будет происходить на: {device}")

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.BCEWithLogitsLoss(reduction='none')

epochs = 3
history_loss = []

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in progress_bar:
        u, i, b, c, y, w = [tensor.to(device) for tensor in batch]
        optimizer.zero_grad()
        y_pred = model(u, i, b, c)
        raw_loss = criterion(y_pred, y)
        weighted_loss = (raw_loss * w).mean()
        weighted_loss.backward()
        optimizer.step()
        total_loss += weighted_loss.item()
        progress_bar.set_postfix({'loss':f"{weighted_loss.item():.4f}"})
    avg_loss = total_loss/len(train_loader)
    history_loss.append(avg_loss)
    print(f"Итог эпохи {epoch+1} | Средний Loss: {avg_loss:.4f}")

Обучение будет происходить на: cuda


Epoch 1/3: 100%|██████████| 782/782 [01:23<00:00,  9.40it/s, loss=0.4204]


Итог эпохи 1 | Средний Loss: 0.4232


Epoch 2/3: 100%|██████████| 782/782 [01:23<00:00,  9.41it/s, loss=0.4369]


Итог эпохи 2 | Средний Loss: 0.4227


Epoch 3/3: 100%|██████████| 782/782 [01:23<00:00,  9.39it/s, loss=0.4254]

Итог эпохи 3 | Средний Loss: 0.4224


In [5]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.1 MB/s eta 0:00:00


In [7]:
from catboost import CatBoostRanker, Pool
import numpy as np
import pandas as pd
import torch

target_user_id = 42
model.eval()

with torch.no_grad():
    u_idx = torch.tensor([target_user_id], device=device)
    u_emb = model.user_emb(u_idx)
    u_repr = F.normalize(model.user_net(u_emb), p=2, dim=1)

    all_i_idx = torch.arange(NUM_ITEMS, device=device)
    all_b_idx = torch.tensor([item_features_dict.get(i.item(), {'brand_idx':0})['brand_idx'] for i in all_i_idx], device=device)
    all_c_idx = torch.tensor([item_features_dict.get(i.item(), {'category_code_idx':0})['category_code_idx'] for i in all_i_idx], device=device)

    i_e = model.item_emb(all_i_idx)
    b_e = model.brand_emb(all_b_idx)
    c_e = model.cat_emb(all_c_idx)

    item_concat = torch.cat([i_e, b_e, c_e], dim=1)
    i_repr = F.normalize(model.item_net(item_concat), p=2, dim=1)

    scores = torch.matmul(u_repr, i_repr.T).squeeze()
    top_100_scores, top_100_items = torch.topk(scores, 100)

top_100_items = top_100_items.cpu().numpy()

ranking_df = pd.DataFrame({
    'user_id': [target_user_id]*100,
    'item_id': top_100_items,
    'item_price': np.random.uniform(10, 1000, 100),
    'user_avg_check': [450.0]*100,
    'item_popularity': np.random.randint(1, 500, 100),
    'two_tower_score': top_100_scores.cpu().numpy(),
    'category': [item_features_dict.get(i, {'category_code_idx':0})['category_code_idx'] for i in top_100_items]
})

ranking_df['target_purchase'] = np.random.choice([0, 1], p=[0.9, 0.1], size=100)
ranker = CatBoostRanker(iterations=50, loss_function='YetiRank', silent=True)
train_pool = Pool(data=ranking_df.drop(columns=['user_id', 'item_id', 'target_purchase']),
                  label=ranking_df['target_purchase'], group_id=ranking_df['user_id'])
ranker.fit(train_pool)
ranking_df['cb_score'] = ranker.predict(train_pool)
ranked_top_50 = ranking_df.sort_values('cb_score', ascending=False).head(50)

def mmr_rerank(items, categories, scores, top_k=10, lambda_param=0.5):
    selected=[]
    selected_cats = set()
    unselected = list(range(len(items)))

    while len(selected) <top_k and unselected:
        best_mmr = -float('inf')
        best_idx = -1

        for idx in unselected:
            rel = scores[idx]
            sim_penalty = 1.0 if categories[idx] in selected_cats else 0.0
            mmr_score = lambda_param*rel-(1-lambda_param)*sim_penalty

            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = idx

        selected.append(items[best_idx])
        selected_cats.add(categories[best_idx])
        unselected.remove(best_idx)

    return selected
final_top_10 = mmr_rerank(
    items=ranked_top_50['item_id'].values,
    categories=ranked_top_50['category'].values,
    scores=ranked_top_50['cb_score'].values,
    top_k=10,
    lambda_param=0.7
)

print("\nИТОГОВАЯ ВЫДАЧА ДЛЯ ЮЗЕРА (TOP-10):")
print("-" * 75)
print(f"{'Ранг':<5} | {'Категория':<25} | {'Бренд':<15} | {'Ориг. ID товара'}")
print("-" * 75)

for rank, item_idx in enumerate(final_top_10, 1):
    cat_idx = item_features_dict.get(item_idx, {'category_code_idx': 0})['category_code_idx']
    brand_idx = item_features_dict.get(item_idx, {'brand_idx': 0})['brand_idx']

    cat_name = encoders['category_code'].inverse_transform([cat_idx])[0]
    brand_name = encoders['brand'].inverse_transform([brand_idx])[0]
    orig_product_id = encoders['product_id'].inverse_transform([item_idx])[0]

    brand_display = brand_name.title() if brand_name != 'unknown_brand' else 'Без бренда'
    cat_display = cat_name if cat_name != 'unknown_category' else 'Разное'

    print(f"#{rank:<4} | {cat_display:<25} | {brand_display:<15} | {orig_product_id}")


ИТОГОВАЯ ВЫДАЧА ДЛЯ ЮЗЕРА (TOP-10):
---------------------------------------------------------------------------
Ранг  | Категория                 | Бренд           | Ориг. ID товара
---------------------------------------------------------------------------
#1    | appliances.environment.vacuum | Samsung         | 3700338
#2    | computers.notebook        | Apple           | 1307054
#3    | construction.tools.drill  | Без бренда      | 12300396
#4    | appliances.environment.vacuum | Samsung         | 3700766
#5    | electronics.smartphone    | Vivo            | 1004937
#6    | Разное                    | Samsung         | 5100767
#7    | electronics.smartphone    | Xiaomi          | 1004741
#8    | electronics.smartphone    | Samsung         | 1004836
#9    | electronics.smartphone    | Xiaomi          | 1004795
#10   | electronics.smartphone    | Samsung         | 1004873
